In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import AIMessage,SystemMessage,HumanMessage
from pydantic import BaseModel
from typing import Literal


In [2]:
load_dotenv()

True

In [3]:
model = ChatGroq(model="llama-3.3-70b-versatile")

## State of WorkFlow

In [4]:
class State:
    usr_msg: str
    msg_category: str

## Node 1 - Classify Message

### Pydantic Schema

In [5]:
class messageClassification(BaseModel):
    category : Literal["booking","faq","reschedule","other"]
    confidence: Literal["low","high"]

### Making LLM aware of schema

In [6]:
structured_model = model.with_structured_output(messageClassification,strict = True)

### Defining Node

In [8]:
def message_classification_node(state: State):
    message = state['usr_msg']
    message_classification_prompt = """You are an intent classifier for a clinic/salon WhatsApp receptionist agent.
        Classify the customer's message into EXACTLY ONE of these categories:

        - booking: customer wants to schedule a NEW appointment
        - faq: customer is asking a question (timings, pricing, location, services offered, doctor availability)
        - reschedule: customer wants to change or cancel an EXISTING appointment
        - other: greetings, unclear messages, or anything not covered above (will be escalated to human staff)

        Rules:
        - If the message mixes a greeting with a request (e.g. "Hi, mujhe appointment chahiye"), classify by the REQUEST, not the greeting.
        - If the message is ambiguous between booking and faq (e.g. "facial ka kya rate hai, book bhi karna hai"), classify as booking since that's the actionable intent.
        - Cancellation requests count as reschedule, not other.
        - Respond with ONLY the category word. No explanation, no punctuation.

        Examples:
        Message: "Kal 3 baje slot mil sakta hai facial ke liye?"
        Category: booking

        Message: "Aap log kitne baje tak khule hote hain?"
        Category: faq

        Message: "Mera appointment kal ka tha, main aaj aana chahta hoon"
        Category: reschedule

        Message: "Mera appointment cancel kar dein please"
        Category: reschedule

        Message: "Salam"
        Category: other

        Message: "Facial ka price kya hai aur kal available hai kya?"
        Category: booking

        Now classify this message:
        Message: "{message}"
        Category:"""

    result = structured_model.invoke( [
            SystemMessage(content = message_classification_prompt),
            HumanMessage(content=message)
            ]
            
        )
    state['msg_Category'] = result.category if result.confidence == "high" else "other"
    return state
    